Name: James Fisher
Date: March 29, 2026
Course: DDS-8555 (Predictive Analytics)
Assignment: Week 5, Applied Question (AQ) 13

In [3]:
## Applied Question 13 in ISLR

# Load libraries
import numpy as np
import pandas as pd

from ISLP import load_data

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Set display options
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)


# Load data
Boston = load_data("Boston").copy()
Boston = Boston.dropna().reset_index(drop=True)

print(Boston.head())
print(Boston.dtypes)


      crim    zn  indus  chas    nox     rm   age     dis  rad  tax  ptratio  lstat  medv
0  0.00632  18.0   2.31     0  0.538  6.575  65.2  4.0900    1  296     15.3   4.98  24.0
1  0.02731   0.0   7.07     0  0.469  6.421  78.9  4.9671    2  242     17.8   9.14  21.6
2  0.02729   0.0   7.07     0  0.469  7.185  61.1  4.9671    2  242     17.8   4.03  34.7
3  0.03237   0.0   2.18     0  0.458  6.998  45.8  6.0622    3  222     18.7   2.94  33.4
4  0.06905   0.0   2.18     0  0.458  7.147  54.2  6.0622    3  222     18.7   5.33  36.2
crim       float64
zn         float64
indus      float64
chas         int64
nox        float64
rm         float64
age        float64
dis        float64
rad          int64
tax          int64
ptratio    float64
lstat      float64
medv       float64
dtype: object


In [4]:
# Create Response Variable
crime_median = Boston["crim"].median()

Boston["danger"] = np.where(Boston["crim"] <= crime_median, 1, 2)

print("Median crime rate:", crime_median)
print(Boston["danger"].value_counts().sort_index())
print(Boston[["crim", "danger"]].head())

Median crime rate: 0.25651
danger
1    253
2    253
Name: count, dtype: int64
      crim  danger
0  0.00632       1
1  0.02731       1
2  0.02729       1
3  0.03237       1
4  0.06905       1


In [5]:
# Define X and y for models, also dropping 'crim' to prevent leakage (since 'danger' is derived from 'crim')
X = Boston.drop(columns=["crim", "danger"])
y = Boston["danger"]

print("Feature columns:", X.columns.tolist())
print(X.shape, y.shape)

Feature columns: ['zn', 'indus', 'chas', 'nox', 'rm', 'age', 'dis', 'rad', 'tax', 'ptratio', 'lstat', 'medv']
(506, 12) (506,)


In [6]:
# Perform train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.30,
    random_state=124,
    stratify=y
)

print("Training set shape:", X_train.shape)
print("Test set shape:", X_test.shape)
print("Training class balance:\n", y_train.value_counts(normalize=True).sort_index())
print("Test class balance:\n", y_test.value_counts(normalize=True).sort_index())

Training set shape: (354, 12)
Test set shape: (152, 12)
Training class balance:
 danger
1    0.5
2    0.5
Name: proportion, dtype: float64
Test class balance:
 danger
1    0.5
2    0.5
Name: proportion, dtype: float64


In [7]:
# Partition predictor subsets (as per directions)
feature_sets = {
    "all_predictors": list(X.columns),

    # common high-signal variables for crime-related separation in Boston
    "strong_structural": ["nox", "dis", "rad", "tax", "ptratio", "lstat", "medv"],

    # smaller subset
    "compact": ["rad", "tax", "lstat", "medv"],

    # neighborhood / housing-oriented subset
    "housing_demo": ["zn", "rm", "age", "ptratio", "lstat", "medv"],

    # accessibility / urbanization subset
    "urban_access": ["indus", "nox", "dis", "rad", "tax"]
}

for name, cols in feature_sets.items():
    print(name, "->", cols)

all_predictors -> ['zn', 'indus', 'chas', 'nox', 'rm', 'age', 'dis', 'rad', 'tax', 'ptratio', 'lstat', 'medv']
strong_structural -> ['nox', 'dis', 'rad', 'tax', 'ptratio', 'lstat', 'medv']
compact -> ['rad', 'tax', 'lstat', 'medv']
housing_demo -> ['zn', 'rm', 'age', 'ptratio', 'lstat', 'medv']
urban_access -> ['indus', 'nox', 'dis', 'rad', 'tax']


In [8]:
# Define helpers (to accelerate and standardize model fitting and evaluation)
def evaluate_models_for_feature_set(X_train, X_test, y_train, y_test, features, set_name):
    results = []

    Xtr = X_train[features]
    Xte = X_test[features]

    models = {
        "Logistic Regression": Pipeline([
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(max_iter=5000))
        ]),

        "LDA": Pipeline([
            ("scaler", StandardScaler()),
            ("model", LinearDiscriminantAnalysis())
        ]),

        "Naive Bayes": Pipeline([
            ("scaler", StandardScaler()),
            ("model", GaussianNB())
        ])
    }

    # fit the three fixed models
    for model_name, model in models.items():
        model.fit(Xtr, y_train)
        preds = model.predict(Xte)
        acc = accuracy_score(y_test, preds)

        results.append({
            "feature_set": set_name,
            "model": model_name,
            "accuracy": acc,
            "features_used": features
        })

        print("=" * 70)
        print(f"{set_name} | {model_name}")
        print("Accuracy:", round(acc, 4))
        print("Confusion matrix:")
        print(confusion_matrix(y_test, preds, labels=[1, 2]))
        print("\nClassification report:")
        print(classification_report(y_test, preds, labels=[1, 2]))

    # KNN with tuning
    knn_pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("model", KNeighborsClassifier())
    ])

    param_grid = {
        "model__n_neighbors": list(range(1, 26))
    }

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    knn_grid = GridSearchCV(
        estimator=knn_pipe,
        param_grid=param_grid,
        cv=cv,
        scoring="accuracy",
        n_jobs=-1
    )

    knn_grid.fit(Xtr, y_train)
    best_knn = knn_grid.best_estimator_
    knn_preds = best_knn.predict(Xte)
    knn_acc = accuracy_score(y_test, knn_preds)

    results.append({
        "feature_set": set_name,
        "model": f"KNN (k={knn_grid.best_params_['model__n_neighbors']})",
        "accuracy": knn_acc,
        "features_used": features
    })

    print("=" * 70)
    print(f"{set_name} | KNN")
    print("Best k:", knn_grid.best_params_["model__n_neighbors"])
    print("CV accuracy:", round(knn_grid.best_score_, 4))
    print("Test accuracy:", round(knn_acc, 4))
    print("Confusion matrix:")
    print(confusion_matrix(y_test, knn_preds, labels=[1, 2]))
    print("\nClassification report:")
    print(classification_report(y_test, knn_preds, labels=[1, 2]))

    return results

In [9]:
# Run all models across all predictor subsets
all_results = []

for set_name, features in feature_sets.items():
    res = evaluate_models_for_feature_set(
        X_train, X_test, y_train, y_test, features, set_name
    )
    all_results.extend(res)

results_df = pd.DataFrame(all_results).sort_values(
    by=["accuracy", "feature_set"],
    ascending=[False, True]
).reset_index(drop=True)

results_df

all_predictors | Logistic Regression
Accuracy: 0.9013
Confusion matrix:
[[73  3]
 [12 64]]

Classification report:
              precision    recall  f1-score   support

           1       0.86      0.96      0.91        76
           2       0.96      0.84      0.90        76

    accuracy                           0.90       152
   macro avg       0.91      0.90      0.90       152
weighted avg       0.91      0.90      0.90       152

all_predictors | LDA
Accuracy: 0.875
Confusion matrix:
[[75  1]
 [18 58]]

Classification report:
              precision    recall  f1-score   support

           1       0.81      0.99      0.89        76
           2       0.98      0.76      0.86        76

    accuracy                           0.88       152
   macro avg       0.89      0.88      0.87       152
weighted avg       0.89      0.88      0.87       152

all_predictors | Naive Bayes
Accuracy: 0.8421
Confusion matrix:
[[68  8]
 [16 60]]

Classification report:
              precision   

,feature_set,model,accuracy,features_used
0,strong_structural,KNN (k=3),0.973684,"[nox, dis, rad, tax, ptratio, lstat, medv]"
1,urban_access,KNN (k=3),0.973684,"[indus, nox, dis, rad, tax]"
2,all_predictors,KNN (k=6),0.927632,"[zn, indus, chas, nox, rm, age, dis, rad, tax,..."
3,all_predictors,Logistic Regression,0.901316,"[zn, indus, chas, nox, rm, age, dis, rad, tax,..."
4,compact,KNN (k=3),0.881579,"[rad, tax, lstat, medv]"
5,strong_structural,Logistic Regression,0.881579,"[nox, dis, rad, tax, ptratio, lstat, medv]"
6,all_predictors,LDA,0.875000,"[zn, indus, chas, nox, rm, age, dis, rad, tax,..."
7,urban_access,Logistic Regression,0.875000,"[indus, nox, dis, rad, tax]"
8,urban_access,LDA,0.868421,"[indus, nox, dis, rad, tax]"
9,strong_structural,LDA,0.861842,"[nox, dis, rad, tax, ptratio, lstat, medv]"


In [10]:
# Display ranking of models
results_df[["feature_set", "model", "accuracy"]]

,feature_set,model,accuracy
0,strong_structural,KNN (k=3),0.973684
1,urban_access,KNN (k=3),0.973684
2,all_predictors,KNN (k=6),0.927632
3,all_predictors,Logistic Regression,0.901316
4,compact,KNN (k=3),0.881579
5,strong_structural,Logistic Regression,0.881579
6,all_predictors,LDA,0.875000
7,urban_access,Logistic Regression,0.875000
8,urban_access,LDA,0.868421
9,strong_structural,LDA,0.861842
